# Codex CLI 0.171.0 업데이트 정리: 기본 모델이 GPT-6.2 Sol로 바뀐 점, Astra로 바꾸는 법, 원격 세션(codex exec --remote, /remote)과 TUI 작업 공간 사용법

위에서부터 차례로 실행해 보세요.

### 0.170.0 설치 후 0.171.0 으로 업데이트

In [ ]:
%%bash
npm i -g @openai/codex@0.170.0 && codex --version
npm i -g @openai/codex@latest && codex --version
codex features list | grep -E 'tui_workspace|tui_extensions_statusline'

### 모델 카탈로그에서 Sol·Astra 확인 (로그인 불필요)

In [ ]:
%%bash
codex debug models | jq -r '.models[] | select(.visibility=="list") | [.slug,.display_name,.description] | @tsv' | head -3

### config.toml 최상위에 model 지정 후 카탈로그·설정 확인

In [ ]:
%%bash
mkdir -p ~/.codex && printf 'model = "gpt-6.2-astra"\n\n' | cat - ~/.codex/config.toml 2>/dev/null > /tmp/c.toml; mv /tmp/c.toml ~/.codex/config.toml
cat ~/.codex/config.toml
codex debug models | jq -r '.models[0:2][] | .slug'

### mock 모델 서버 파일 만들기

In [ ]:
%%bash
mkdir -p /work/scripts && cat > /work/scripts/mock.py <<'E'
import json
from http.server import BaseHTTPRequestHandler, HTTPServer
n=[0]
def ev(t,d): return f"event: {t}\ndata: {json.dumps({'type':t,**d})}\n\n"
class H(BaseHTTPRequestHandler):
    def log_message(self,*a): pass
    def do_POST(self):
        req=json.loads(self.rfile.read(int(self.headers.get('content-length',0))) or b'{}')
        n[0]+=1
        has_out=any(i.get('type')=='function_call_output' for i in req.get('input',[]) if isinstance(i,dict))
        rid=f'resp_{n[0]}'
        out=[ev('response.created',{'response':{'id':rid}})]
        if not has_out:
            item={'type':'function_call','id':f'fc_{n[0]}','call_id':f'call_{n[0]}','name':'exec_command','arguments':json.dumps({'cmd':"printf 'print(\"hello\")\\n' > hello.py && cat hello.py"})}
        else:
            item={'type':'message','id':f'm_{n[0]}','role':'assistant','content':[{'type':'output_text','text':'Created hello.py (mock model).'}]}
        out.append(ev('response.output_item.done',{'item':item}))
        out.append(ev('response.completed',{'response':{'id':rid,'usage':{'input_tokens':1,'output_tokens':1,'total_tokens':2}}}))
        self.send_response(200); self.send_header('content-type','text/event-stream'); self.end_headers()
        self.wfile.write(''.join(out).encode())
HTTPServer(('127.0.0.1',9998),H).serve_forever()
E
mkdir -p /work/mock/home && cat > /work/mock/home/config.toml <<'E'
model = "gpt-5.1-codex"
model_provider = "mock"
[model_providers.mock]
name = "mock"
base_url = "http://127.0.0.1:9998/v1"
wire_api = "responses"
requires_openai_auth = false
E
npm i -g @openai/codex@0.171.0 && codex --version

### app server 띄우고 codex exec --remote 로 hello.py 만들기

In [ ]:
%%bash
set -e
export CODEX_HOME=/work/mock/home
(nohup python3 /work/scripts/mock.py > /work/mock.log 2>&1 &)
rm -rf /work/mock/e3; mkdir -p /work/mock/e3; cd /work/mock/e3
(nohup codex app-server -c sandbox_mode='"danger-full-access"' -c approval_policy='"never"' --listen ws://127.0.0.1:4500 > /work/appserver3.log 2>&1 &)
sleep 4
codex exec --remote ws://127.0.0.1:4500 "빈 폴더에 hello.py 만들기"
ls -la; cat hello.py